<a href="https://colab.research.google.com/github/Sriyansh-36-AI-NITJ/Concurrent-Programming-Lab/blob/main/Count_pixels_End_Sem_Practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install cupy-cuda12x numba scikit-learn

In [14]:
import numpy as np
import cupy as cp
from numba import cuda
import time
from tensorflow.keras.datasets import mnist

In [15]:
print("Importing the MNIST dataset...")

# Load dataset (downloads a lightweight .npz file)
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# Cast the data to float32 for GPU processing
images = train_images.astype(np.float32)

# Keras natively loads them as 2D spatial images (60000, 28, 28)
THRESHOLD = 127.0

# We will process the entire training set of 60,000 images
batch_images = images
print(f"Dataset successfully imported! Processing shape: {batch_images.shape}")

Importing the MNIST dataset...
Dataset successfully imported! Processing shape: (60000, 28, 28)


In [16]:
def count_with_cupy(images_np, threshold):
    # 1. Transfer data from CPU to GPU
    d_images = cp.asarray(images_np)

    start_time = time.time()

    # 2. Vectorized computation on GPU
    # Creates a boolean mask and sums across the spatial dimensions (axis 1 and 2)
    counts = cp.sum(d_images > threshold, axis=(1, 2))

    # 3. Synchronize to ensure accurate timing measurement
    cp.cuda.Stream.null.synchronize()
    end_time = time.time()

    # 4. Bring results back to CPU
    return counts.get(), end_time - start_time

In [17]:
@cuda.jit
def count_above_threshold_kernel(images, threshold, counts):
    # Extract 3D grid coordinates
    img_idx, row, col = cuda.grid(3)

    # Boundary checks
    if (img_idx < images.shape[0] and
        row < images.shape[1] and
        col < images.shape[2]):

        # Check pixel intensity
        if images[img_idx, row, col] > threshold:
            # Atomic add prevents race conditions between threads
            cuda.atomic.add(counts, img_idx, 1)

def count_with_numba(images_np, threshold):
    N, rows, cols = images_np.shape

    # Move data to GPU
    d_images = cuda.to_device(images_np)
    d_counts = cuda.to_device(np.zeros(N, dtype=np.int32))

    # Define thread hierarchy (8x8x8 = 512 threads per block)
    threads_per_block = (8, 8, 8)

    # Calculate grid dimensions
    blocks_per_grid_z = int(np.ceil(N / threads_per_block[0]))
    blocks_per_grid_y = int(np.ceil(rows / threads_per_block[1]))
    blocks_per_grid_x = int(np.ceil(cols / threads_per_block[2]))
    blocks_per_grid = (blocks_per_grid_z, blocks_per_grid_y, blocks_per_grid_x)

    start_time = time.time()

    # Execute Kernel
    count_above_threshold_kernel[blocks_per_grid, threads_per_block](d_images, threshold, d_counts)
    cuda.synchronize()

    end_time = time.time()

    return d_counts.copy_to_host(), end_time - start_time

In [18]:
# Clear out any conflicting cupy installations
!pip uninstall -y cupy cupy-cuda12x cupy-cuda11x

# Reinstall the correct version for Colab's current CUDA environment
!pip install cupy-cuda12x numba

Found existing installation: cupy-cuda12x 14.1.1
Uninstalling cupy-cuda12x-14.1.1:
  Successfully uninstalled cupy-cuda12x-14.1.1
  Using cached cupy_cuda12x-14.1.1-cp312-cp312-manylinux2014_x86_64.whl.metadata (2.8 kB)
Using cached cupy_cuda12x-14.1.1-cp312-cp312-manylinux2014_x86_64.whl (133.5 MB)


In [19]:
print("Warming up GPU context...")
# Warm-up calls to compile the kernels and initialize memory contexts
_, _ = count_with_cupy(batch_images[:10], THRESHOLD)
_, _ = count_with_numba(batch_images[:10], THRESHOLD)

print("\nRunning benchmarks on 60,000 images...")

# Run CuPy
cupy_counts, cupy_time = count_with_cupy(batch_images, THRESHOLD)
print(f"CuPy execution time:  {cupy_time:.6f} seconds")

# Run Numba
numba_counts, numba_time = count_with_numba(batch_images, THRESHOLD)
print(f"Numba execution time: {numba_time:.6f} seconds")

# Verify accuracy
match = np.array_equal(cupy_counts, numba_counts)
print(f"\nResults match perfectly: {match}")
print(f"Sample pixel counts (first 5 images): {cupy_counts[:5]}")

Warming up GPU context...

Running benchmarks on 60,000 images...
CuPy execution time:  0.017619 seconds
Numba execution time: 0.003863 seconds

Results match perfectly: True
Sample pixel counts (first 5 images): [111 125  81  66  91]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 32 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [20]:
# 1. Warm up the GPU (compiles kernels, initializes context)
print("Warming up GPU...")
_, _ = count_with_cupy(batch_images[:10], THRESHOLD)
_, _ = count_with_numba(batch_images[:10], THRESHOLD)

# 2. Run and time CuPy
cupy_counts, cupy_time = count_with_cupy(batch_images, THRESHOLD)
print(f"\nCuPy time:  {cupy_time:.6f} seconds")

# 3. Run and time Numba
numba_counts, numba_time = count_with_numba(batch_images, THRESHOLD)
print(f"Numba time: {numba_time:.6f} seconds")

# 4. Verification
match = np.array_equal(cupy_counts, numba_counts)
print(f"\nResults match: {match}")
print(f"Sample counts (first 5 images): {cupy_counts[:5]}")

Warming up GPU...

CuPy time:  0.017638 seconds
Numba time: 0.003880 seconds

Results match: True
Sample counts (first 5 images): [111 125  81  66  91]
